# SIOPE education finance: how the raw layer is obtained

**Publisher:** Fundo Nacional de Desenvolvimento da Educação (FNDE)  
**Official source:** <https://www.fnde.gov.br/siope/>

Education revenue, expenditure and indicators reported by municipalities and states, from the SIOPE open-data API (Olinda), the FNDE SIOPE exports, and the budget execution report (RREO) PDFs parsed into tables.

**Grain and keys:** Municipality (or state) by year and reporting period. Indicator rows repeat by `num_periodo`, so analyses keep the latest period per municipality and year. Municipality joins to IBGE through `codigo_municipio`.

**Released as:**

- `siope-analytics`: semantic (16 tables)
- `siope-raw-trusted-part-1`: raw (2 tables)
- `siope-raw-trusted-part-2`: raw (13 tables)
- `siope-raw-trusted-part-3`: raw, trusted (8 tables)
- `siope-raw-trusted-part-4`: trusted (10 tables)

Every table and column is documented in the [interactive data map](https://lucas.rangeltech.net/datamap/). Source code and the other notebooks: [https://github.com/LucasRangelSSouza/brazil-public-data-map](https://github.com/LucasRangelSSouza/brazil-public-data-map).

## Status of this notebook

Executed against the live source when this notebook was written; the outputs below are from that run.

In [1]:
import json, time
import requests

HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; public-data-map-notebook; +https://github.com/LucasRangelSSouza/brazil-public-data-map)"}

def get(url, retries=4, timeout=90, **kw):
    """GET with exponential backoff. Public portals answer 429 or reset connections under load."""
    for attempt in range(retries):
        try:
            r = requests.get(url, headers=HEADERS, timeout=timeout, **kw)
            if r.status_code in (429, 500, 502, 503, 504):
                raise requests.HTTPError(f"HTTP {r.status_code}")
            r.raise_for_status()
            return r
        except requests.RequestException as exc:
            if attempt == retries - 1:
                raise
            time.sleep(2 ** attempt)

## 1. The open-data service

The service is OData. Each entity is a function that takes the reporting year, the period (bimester) and the state, so a query is a path and not a `$filter`. The metadata document lists the eight entities and their parameters.

In [2]:
import re
BASE = "https://www.fnde.gov.br/olinda-ide/servico/DADOS_ABERTOS_SIOPE/versao/v1/odata"
meta = get(f"{BASE}/$metadata").text
for name, body in re.findall(r'<Function Name="([^"]+)"[^>]*>(.*?)</Function>', meta, re.S):
    print(name, re.findall(r'<Parameter Name="([^"]+)"', body))

Dados_Gerais_Siope_Dados_Responsaveis ['Ano_Consulta', 'Num_Peri', 'Sig_UF']
Indicadores_Siope ['Ano_Consulta', 'Num_Peri', 'Sig_UF']
Informacoes_Complementares_Siope ['Ano_Consulta', 'Num_Peri', 'Sig_UF']
Receita_Siope ['Ano_Consulta', 'Num_Peri', 'Sig_UF']
Dados_Gerais_Siope ['Ano_Consulta', 'Num_Peri', 'Sig_UF']
Despesas_Siope ['Ano_Consulta', 'Num_Peri', 'Sig_UF']
Remuneracao_Siope ['Ano_Declaracao', 'Num_Peri', 'Mes_Exercicio', 'Sig_UF']
Despesas_Funcao_Educacao_Siope ['Ano_Consulta', 'Num_Peri', 'Sig_UF']


## 2. One entity, one state, one period

The state is the federal-unit code; for the municipalities of a state the rows carry `COD_MUNI`. Indicator rows repeat by period, which is why analyses keep the latest `NUM_PERI` per municipality and year.

In [3]:
url = f"{BASE}/Indicadores_Siope(Ano_Consulta=2023,Num_Peri=6,Sig_UF='DF')?$top=5&$format=json"
rows = get(url).json()["value"]
import pandas as pd
pd.DataFrame(rows)[["TIPO", "NUM_ANO", "NUM_PERI", "SIG_UF", "COD_INDI", "NOM_INDI", "VAL_INDI"]]

,TIPO,NUM_ANO,NUM_PERI,SIG_UF,COD_INDI,NOM_INDI,VAL_INDI
0,Estadual,2023,6,DF,5,Percentual de aplicação em Despesas de Capital...,0
1,Estadual,2023,6,DF,40,Percentual de aplicação das receitas de impost...,27.16
2,Estadual,2023,6,DF,42,Percentual de aplicação do FUNDEF ou FUNDEB na...,87.88
3,Estadual,2023,6,DF,43,Percentual de aplicação do FUNDEF ou FUNDEB em...,7.99
4,Estadual,2023,6,DF,44,Percentual das receitas do FUNDEB não aplicad...,4.12


## 3. Paging and the other entities

The service pages with `$top` and `$skip`. The other entities are `Dados_Gerais_Siope`, `Receita_Siope`, `Despesas_Siope`, `Despesas_Funcao_Educacao_Siope`, `Informacoes_Complementares_Siope`, `Remuneracao_Siope` (which takes `Ano_Declaracao` and `Mes_Exercicio`) and `Dados_Gerais_Siope_Dados_Responsaveis`. They map to the released `api_olinda_siope_*` tables.

In [4]:
def fetch_all(entity, page_size=5000, max_pages=2, **params):
    args = ",".join(f"{k}={v!r}" if isinstance(v, str) else f"{k}={v}" for k, v in params.items())
    out = []
    for page in range(max_pages):
        data = get(f"{BASE}/{entity}({args})?$top={page_size}&$skip={page * page_size}&$format=json").json()["value"]
        out += data
        if len(data) < page_size:
            break
    return out

receita = fetch_all("Receita_Siope", Ano_Consulta=2023, Num_Peri=6, Sig_UF="DF")
print(len(receita), "revenue rows for DF, 2023, period 6")

1044 revenue rows for DF, 2023, period 6


## 4. The budget execution report (RREO)

The `rreo_siope_*` tables come from the RREO reports, which FNDE publishes as PDFs. They were parsed into tables; the file name in the `arquivo` column identifies the report (`RREO_Municipal_<IBGE code>_<bimester>_<year>.pdf`). The PDFs themselves are not part of the release.

## Compare with the released tables

The released files keep the field names of the source in `raw` and apply consistent typing and snake_case names in `trusted`. Details per column are in the [data map](https://lucas.rangeltech.net/datamap/#/subject/siope).

In [5]:
# Read one released table without downloading the whole dataset. Public datasets need no login.
import kagglehub, pandas as pd
path = kagglehub.dataset_download("lucasrangelss/siope-raw-trusted-part-4", path="trusted__rreo_siope_uf.parquet")
pd.read_parquet(path).head()

C:\Users\lucas.rangel\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  0%|          | 0.00/7.66M [00:00<?, ?B/s]

 13%|█▎        | 1.00M/7.66M [00:00<00:06, 1.06MB/s]

 26%|██▌       | 2.00M/7.66M [00:01<00:02, 2.15MB/s]

 39%|███▉      | 3.00M/7.66M [00:01<00:01, 3.12MB/s]

...

 65%|██████▌   | 5.00M/7.66M [00:01<00:00, 4.79MB/s]

 78%|███████▊  | 6.00M/7.66M [00:01<00:00, 5.54MB/s]

 91%|█████████▏| 7.00M/7.66M [00:01<00:00, 6.18MB/s]

100%|██████████| 7.66M/7.66M [00:01<00:00, 4.17MB/s]

,co_uf,sg_uf,ano,bimestre,secao,codigo,nivel,descricao,coluna,posicao,nome_metrica,valor,arquivo,ftp_mod_date,gcs_uri_pdf,data_extracao
0,53,None,2026,1,19,19,0,TOTAL DAS DESPESAS CUSTEADAS COM SUPERAVIT DO ...,COL_x,6,% LIQUIDADO (g/d),3.136110e+07,RREO_DF_53_1_2026.pdf,2026-03-30T16:45:00,gs://mi-prd-lake-landing-zone/siope/rreo/raw/u...,2026-07-10 22:16:04.953009+00:00
1,53,None,2026,1,18,18,0,TOTAL DA RECEITA RECEBIDA E NAO APLICADA NO EX...,COL_p,3,COL_u,0.000000e+00,RREO_DF_53_1_2026.pdf,2026-03-30T16:45:00,gs://mi-prd-lake-landing-zone/siope/rreo/raw/u...,2026-07-10 22:16:04.953009+00:00
2,53,None,2026,1,30,30.2,1,Executadas com Recursos do FUNDEB Impostos,SALDO FINAL RP,5,% EMPENHADO (e/d),9.537331e+07,RREO_DF_53_1_2026.pdf,2026-03-30T16:45:00,gs://mi-prd-lake-landing-zone/siope/rreo/raw/u...,2026-07-10 22:16:04.953009+00:00
3,53,None,2026,1,15,15,0,MINIMO DE 70% DO FUNDEB NA REMUNERACAO DOS PRO...,COL_j,1,DOTACAO INICIAL,4.500363e+08,RREO_DF_53_1_2026.pdf,2026-03-30T16:45:00,gs://mi-prd-lake-landing-zone/siope/rreo/raw/u...,2026-07-10 22:16:04.953009+00:00
4,53,None,2026,1,19,19,0,TOTAL DAS DESPESAS CUSTEADAS COM SUPERAVIT DO ...,COL_w,5,INSCRITAS EM RP NAO PROCESSADOS,3.136110e+07,RREO_DF_53_1_2026.pdf,2026-03-30T16:45:00,gs://mi-prd-lake-landing-zone/siope/rreo/raw/u...,2026-07-10 22:16:04.953009+00:00
